# **Lab 1: Edge AI Inference on Arm Using the ExecuTorch Runtime + XNNPACK and Kleidi AI**
### CPU Inference on Raspberry Pi 5 (Arm-powered) - PyTorch vs ExecuTorch at the Edge

## **Introduction**

Welcome to the **Edge AI Inference on Arm Using the ExecuTorch Runtime** lab! In this hands-on session, you will explore how PyTorch models can be exported and lowered to the ExecuTorch runtime, particularly via XNNPACK and Kleidi AI, for effective, lightweight, minimalist inference at the Edge. 

You will understand how ExecuTorch can be utilized to reduce the memory footprint, the energy per inference, and the latency, which are critical for many deployments at the Edge on constrained, low-power, lower-specification devices. This lab will showcase when it is appropriate to utilize ExecuTorch, and will highlight important caveats when deciding between ExecuTorch and PyTorch at the Edge.

**Requirements**: To complete this lab, you are recommended to utilize a Raspberry Pi 5 or similar Arm-powered Edge device running a Linux-based OS. You can utilize higher-specification devices. For example, this has been tested on a MacBook Pro M4, however the benefits of ExecuTorch over PyTorch are intended for Edge and embedded devices and can be obscured or lost on higher-spec devices. Since PyTorch is well optimized for a device like the MacBook M4, performance results comparing PyTorch and ExecuTorch may be inverted.

### **Lab Objectives:** 

The objectives of this lab are as follows:

1. **Understand Edge AI and TinyML**:
   - Understand the concept of AI inference at the Edge, what ExecuTorch is, and the potential of CPU inference.

2. **PyTorch and ExecuTorch Inference**:
   - Perform inference using transformer-based AI/ML models (encoder and decoder) both with the PyTorch and ExecuTorch runtimes.
   - Export and lower from PyTorch to ExecuTorch using the ExecuTorch portable library
   - Understand why XNNPACK, integrated with Arm KleidiAI, is typically used as a backend for optimal Arm-based CPU inference and how PyTorch models can be lowered via XNNPACK.

3. **Benchmark PyTorch vs ExecuTorch**:
   - Measure latency for each runtime (comparing PyTorch, ExecuTorch Portable, ExecuTorch + XNNPACK), and understand the additional benefits of ExecuTorch beyond latency and throughput, including portability, lack of dependencies, memory footprint, etc.

4. **Identify next steps for learning more about Edge AI and ExecuTorch**:
   - Understand the importance of choosing appropriate AI/ML models and correctly leveraging the benefits ExecuTorch provides.
   - Understand the the appropriate deployment scenarios for utilizing the ExecuTorch runtime - where it typically provides benefits and where it doesn't.
   - Be equipped to move onto the next lab, which will introduce quantization for a typical image classification application.

### **Prerequisites**

To follow this lab, you should have a basic understanding of:

- **Python programming**: Experience writing and running Python scripts.
- **AI/ML principles**: Basic understanding of AI/ML and model types.

> Note: It is assumed that the same Jupyter Kernel, is used throughout this lab and that all cells are run sequentially.

### **Recommended**
- Non-essential but recommended prior to this lab is to complete the [Optimizing GenAI on Arm](https://www.edx.org/learn/computer-science/arm-education-ai-on-arm) course on edX or Coursera.

### **What is Edge AI / TinyML?**

Edge AI refers to deploying machine learning models on small, low-power devices with stricter constraints on memory, compute, and energy. Key strategies in Edge AI include using efficient model architectures and techniques like **quantization** (e.g., reducing precision from 32-bit floats to 8-bit integers) to cut down computation and memory requirements, as described in Lab 2. Edge AI is typically deployed on dedicated neural processing units (NPUs), such as the Arm Ethos-U, or on CPUs, as used in this lab. Deployment on CPU is possible due to the optimizations above. Due to the ubiquitous nature of CPUs, as every general-purpose compute-enabled device contains a CPU, inference can be performed on a wide range of devices.

Another key aspect of Edge AI is privacy. Sensitive data, for example camera images and microphone audio, can be processed locally by the model rather than being sent to a cloud API. This is inherently more private and secure. For applications such as smart cameras, voice assistants, or health monitors, on-device inference ensures personal data never leaves the user’s control. Another benefit is that there is no need for an internet connection, which is ideal for devices deployed in environments where an internet connection may not be feasible and also opens up a range of device options at lower specification, price-point, and capability.

### **What are PyTorch and ExecuTorch?**

**PyTorch** is a popular open-source machine learning framework for building and training deep learning models. Originally developed by Meta’s AI Research lab (FAIR) in 2016, PyTorch has become a cornerstone of AI research and production thanks to its flexible, Pythonic design. It offers dynamic computation graphs, known as eager execution, an intuitive API, and a vast ecosystem, including TorchVision, Hugging Face Transformers, and others that make it easy for engineers to prototype and train neural networks. In essence, PyTorch is widely used to develop models on powerful hardware, such as GPUs and CPUs, accelerating the path from research prototyping to production deployments.

**ExecuTorch**, on the other hand, is officially described as "PyTorch’s unified solution for deploying AI models on-device, from smartphones to microcontrollers, built for privacy, performance, and portability.” It is an inference runtime and toolchain that takes a PyTorch model and prepares it for efficient on-device execution. Importantly, ExecuTorch is PyTorch-native. It uses the same APIs and requires no translation to another format, unlike exporting to ONNX or TFLite. Because ExecuTorch is a part of the PyTorch ecosystem, developers can use familiar workflows. The model’s behavior and accuracy remain consistent between PyTorch and ExecuTorch since there’s no translation to a different graph format that might introduce errors.

### **Why ExecuTorch?**

Those familiar with PyTorch will likely be aware that PyTorch, without ExecuTorch, can be deployed at the Edge. This provides some of the benefits of Edge AI discussed above. So when is it appropriate to use ExecuTorch, and when should you stick with PyTorch?

The core ExecuTorch runtime is extremely lightweight, in the order of ~50KB for the base component. This minimalist C++ runtime includes just the essentials needed for model execution. In contrast, the full PyTorch library is hundreds of megabytes. The small size means less memory usage on device, which is critical for low-memory devices, and faster startup times. As noted in the official documentation, ExecuTorch’s “tiny runtime” can run on everything from microcontrollers to high-end smartphones. In practical terms, this means you can integrate AI into very constrained environments that couldn’t possibly accomodate the full PyTorch.

- **Lower Memory Usage at Inference**: Beyond just code size, ExecuTorch also tends to have a lower dynamic memory footprint when running models. By pre-compiling and optimizing the model ahead-of-time, it can eliminate overhead that an eager PyTorch model might incur, such as extra tensor allocations and Python objects. For example, an LLM that might barely fit in 2GB with PyTorch might comfortably run in 1.5GB with ExecuTorch due to memory optimizations and the elimination of Python interpreter overhead.

- **Improved Inference Speed and Throughput**: ExecuTorch leverages ahead-of-time (AOT) compilation and optimized kernels to boost runtime performance. When you export a model via torch.export, it captures a static graph. ExecuTorch then applies graph transformations and optimizations, such as operator fusion, and selects the best kernels for the target CPU. On Arm CPUs, the best approach is typically to leverage the XNNPACK library (highly optimized neural network kernels) as the backend. In ExecuTorch 1.0, XNNPACK is tuned by Arm’s Kleidi AI to exploit the available hardware extensions on the Arm CPU. In practice, that means lower latency per inference and higher throughput, measured as inferences per second, compared with using the default ExecuTorch configration and compared with PyTorch. For example, Meta found that after migrating various on-device models, including vision and language models, to ExecuTorch, they saw significant speedups. Features in Instagram, WhatsApp, and Messenger experienced faster inference and reduced load times [[1]](https://engineering.fb.com/2025/07/28/android/executorch-on-device-ml-meta-family-of-apps/#:~:text=Cutouts%20is%20one%20of%20Instagram%E2%80%99s,DAU).

- **Power Efficiency**: Faster inference directly translates to energy savings. If the CPU spends less time busy on a task, it can return to idle or low-power states sooner. Moreover, optimized kernels often utilize vector instructions and efficient memory access patterns, doing more work per clock cycle. The net effect is lower energy per inference. Since on-device inference also avoids sending data over wireless networks, which is a further power draw, the overall system power usage for a given task can be much lower than with a cloud-offloaded approach. In summary, by maximizing efficient use of the CPU (and other accelerators), ExecuTorch achieves better performance per watt, which is essential for battery-powered devices.

- **Quantization and Model Optimization**: As in PyTorch, ExecuTorch fully supports quantized models (INT8, etc.), which can drastically improve both speed and memory use. This can give two to four times speed improvements and a fourfold reduction in model size (32-bit to 8-bit). Quantization is especially beneficial for Arm CPUs, which often have specialized instructions for INT8 arithmetic (e.g., DOTPROD extension on NEON, or I8MM). This will be introduced in Lab 2.

To keep the notebook output clean and focus on learning, we will suppress some verbose, non-critical warnings.


In [ ]:
import warnings
import logging
import os

# Suppress Python warnings (including FutureWarning)
warnings.filterwarnings("ignore")

# Reduce logging verbosity
logging.getLogger().setLevel(logging.ERROR)

# Suppress Python warning environment noise
os.environ["PYTHONWARNINGS"] = "ignore"

## **Open Pre-Trained Transformers (OPT) LLM Example - 125M Parameters**

Use pip to install various libraries for the lab:

In [ ]:
%%bash 

pip install torch torchvision executorch transformers psutil matplotlib torchao 

mkdir -p Transformer_Lab_ExecuTorch
cd Transformer_Lab_ExecuTorch

We’ll compare a PyTorch eager baseline against ExecuTorch with no XNNPACK backend and ExecuTorch with the XNNPACK delegate for the OPT-125m model (Open Pre-Trained Transformers) from Meta. We wrap the model to return only last-token logits to keep output small and stable. We’ll also inspect and discuss which operators are delegated to XNNPACK.

We use multiple-iteration benchmarking for PyTorch and ExecuTorch with XNNPACK to ensure fair benchmarking. However, we will only do a single timed run for standard ExecuTorch (no backend optimization) to avoid slow repeated runs. By default the number of iterations (`ITERS`) is set to 10 - you are welcome to change this.

All benchmarks, PyTorch and ExecuTorch, are run in FP32 to ensure fairness. A typical deployment scenario for ExecuTorch would, however, be quantized, such as down to INT8. This will be discussed in the second lab.

Set-up and functions for the benchmarking and display are created below:

In [ ]:
# =========================
# PyTorch vs ExecuTorch vs ExecuTorch+XNNPACK.
# Model: OPT-125m.
# =========================

import os, time, pathlib
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.runtime import Runtime
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

# -------------------------
# Config.
# -------------------------
MODEL_ID = "facebook/opt-125m"
SEQ_LEN = 128
BATCH = 1
DTYPE = torch.float32
DEVICE = "cpu"

WARMUP = 1
ITERS = 20  # PyTorch + ET+XNNPACK benchmark over several runs.

TOKENIZERS_PARALLELISM = True

outdir = pathlib.Path("Transformer_Lab_ExecuTorch/executorch_artifacts")
outdir.mkdir(exist_ok=True)

def bench_dist(fn, warmup=WARMUP, iters=ITERS, timer=time.perf_counter):
    
    # Warmup.
    for _ in range(warmup):
        fn()

    # Timed runs.
    times = np.empty(iters, dtype=np.float64)
    for i in range(iters):
        t0 = timer()
        fn()
        t1 = timer()
        times[i] = (t1 - t0) * 1000.0  # ms

    # Summary stats.
    stats = {
        "count": int(iters),
        "mean_ms": float(times.mean()),
        "median_ms": float(np.median(times)),
        "std_ms": float(times.std(ddof=0)),
        "min_ms": float(times.min()),
        "max_ms": float(times.max()),
        "p50_ms": float(np.percentile(times, 50)),
        "p90_ms": float(np.percentile(times, 90)),
        "p95_ms": float(np.percentile(times, 95)),
        "p99_ms": float(np.percentile(times, 99)),
    }
    return times, stats

def print_bench_report(name, stats, warmup, iters):
    print("\n" + "=" * 50)
    print(f"BENCHMARK RESULTS: {name}")
    print("=" * 50)
    print(f"  Warmup runs: {warmup}")
    print(f"  Number of runs: {iters}")
    print("-" * 50)
    print(f"  Mean:   {stats['mean_ms']:.2f} ms")
    print(f"  Median: {stats['median_ms']:.2f} ms")
    print(f"  Min:    {stats['min_ms']:.2f} ms")
    print(f"  Max:    {stats['max_ms']:.2f} ms")
    print("-" * 50)

def plot_time_distribution(times_ms, name, bins=30):
    
    mean_ms = float(times_ms.mean())

    plt.figure(figsize=(12, 4))

    # Histogram.
    plt.subplot(1, 2, 1)
    plt.hist(times_ms, bins=bins, edgecolor="black", alpha=0.7)
    plt.axvline(mean_ms, color="red", linestyle="--", label=f"Mean: {mean_ms:.2f} ms")
    plt.xlabel("Inference Time (ms)")
    plt.ylabel("Frequency")
    plt.title(f"{name}: Time Distribution")
    plt.legend()
    plt.grid(alpha=0.3)

    # Time series.
    plt.subplot(1, 2, 2)
    plt.plot(times_ms, marker="o", markersize=3, alpha=0.6)
    plt.axhline(mean_ms, color="red", linestyle="--", label=f"Mean: {mean_ms:.2f} ms")
    plt.xlabel("Run Number")
    plt.ylabel("Inference Time (ms)")
    plt.title(f"{name}: Time Over Runs")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

def time_one(fn, warmup=WARMUP):
    
    for _ in range(warmup):
        fn()
    t0 = time.perf_counter()
    fn()
    t1 = time.perf_counter()
    return (t1 - t0) * 1000

print("Torch:", torch.__version__)

We load OPT-125m and create a fixed-length tokenized input (SEQ_LEN=128). Fixed shapes keep export and lowering straightforward and make timing more consistent.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

prompt = "Write a short sentence about fast inference."
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=SEQ_LEN,
)

input_ids = inputs["input_ids"].to(DEVICE)
attention_mask = inputs["attention_mask"].to(DEVICE)

print("input_ids:", tuple(input_ids.shape), input_ids.dtype)
print("attention_mask:", tuple(attention_mask.shape), attention_mask.dtype)

The output above means the model is receiving a single input sequence, `batch_size = 1`, padded or truncated to 128 tokens, where `input_ids` contains the integer token indices (`torch.int64`) and `attention_mask` is a matching 128-length integer mask indicating which token positions are real input (1) versus padding (0).

Transformers produce logits for every position, `[B, T, V]`, during training because we supervise all time steps in parallel and compute a loss for each token. During autoregressive inference, however, only the logits of the final position are needed to generate the next token, since earlier tokens are already fixed and their predictions are no longer used. Returning just `[B, V]` therefore preserves the exact computation required for generation while avoiding unnecessary output materialization. This is the standard technique used in LLM chatbots and production systems that generate text one token at a time. This is done with the class below:

In [ ]:
class LastTokenLogits(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        out = self.m(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
        return out.logits[:, -1, :]  # [B, V].

wrapped = LastTokenLogits(model).eval()

We measure PyTorch eager with warmup + `ITERS` runs and report the average latency.

In [ ]:
def pt_forward():
    with torch.inference_mode():
        _ = wrapped(input_ids, attention_mask)

pt_times, pt_stats = bench_dist(pt_forward, warmup=WARMUP, iters=ITERS)
print_bench_report("PyTorch", pt_stats, WARMUP, ITERS)

`torch.export.export` captures the computation in an `ExportedProgram`, which ExecuTorch lowering consumes.

In [ ]:
example_inputs = (input_ids, attention_mask)
ep = export(wrapped, example_inputs)
print("ExportedProgram captured.")

This is the non-XNNPACK ExecuTorch path. We only do one timed run here (after warmup), because we will quickly move on to using the XNNPACK delegate for reasons that will become clear. Do not be concerned if this cell takes longer to run.


In [ ]:
edge_port = to_edge_transform_and_lower(ep)
et_port = edge_port.to_executorch()

pte_port = outdir / "opt125m_portable.pte"
with open(pte_port, "wb") as f:
    et_port.write_to_file(f)

rt = Runtime.get()
prog_port = rt.load_program(pte_port)
method_port = prog_port.load_method("forward")

def et_port_forward():
    _ = method_port.execute((input_ids, attention_mask))

# Only single timed run (plus warmup).
et_port_ms = time_one(et_port_forward, warmup=WARMUP)
print(f"\nExecuTorch portable single run: {et_port_ms:.2f} ms")
print(f"PyTorch eager average over {ITERS} runs: {pt_stats['mean_ms']:.2f} ms\n\n")

print(f"Portable PTE size (MB): {pte_port.stat().st_size / 1e6:.2f}")

Whilst it may vary from device to device, this initial run of ExecuTorch likely took significantly longer than the average PyTorch eager execution. This may be surprising. Surely ExecuTorch should run faster? You might think this is just a one-off, as we aren't averaging over many runs.

In fact, ExecuTorch portable is often slower than PyTorch because it runs operators using the Portable Kernel Library, which is a reference implementation focused on broad operator coverage and correctness rather than high performance. In contrast, PyTorch dispatches many operations, especially large matrix multiplications in transformer models, to highly optimized CPU backends using Kleidi AI. Kleidi AI is Arm’s open-source library of highly optimized low-level compute kernels for AI workloads, particularly focused on accelerating matrix multiplication and other core operations used in neural networks on Arm CPUs. It provides architecture-tuned implementations that take advantage of features such as NEON and SVE to deliver strong performance and energy efficiency for inference workloads.

Even when latency isn’t better, ExecuTorch can still be the right choice because inference on-device may be constrained more by memory and deployability than by raw speed. PyTorch often isn’t practical on tiny or embedded targets due to its runtime size and dependency footprint, whereas ExecuTorch packages the model into a compact, self-contained .pte program that can be loaded and executed by a lightweight runtime. In other words, the small .pte artifact you generated is a deployable unit that makes it feasible to run the model on devices where the full PyTorch stack simply wouldn’t fit or function reliably, even if the “portable” kernels trade performance for portability and footprint. Although PyTorch appears faster on a relatively powerful edge device like a Raspberry Pi, more constrained devices simply may not be able to support it.

However, we can leverage Kleidi AI on ExecuTorch via XNNPACK to combine both the portability and reduced memory footprint with performance. XNNPACK is a highly optimized open-source library of neural network inference kernels designed primarily for CPU execution. It provides fast implementations of common operators such as convolutions and fully connected (linear) layers.

You can find official PyTorch/ExecuTorch documentation on the XNNPACK backend [here](https://docs.pytorch.org/executorch/1.1/ios-xnnpack.html) and [here](https://docs.pytorch.org/executorch/0.3/native-delegates-executorch-xnnpack-delegate.html).

We can lower via XNNPACK with `XnnpackPartitioner()`. The partitioner finds subgraphs it can offload to XNNPACK and rewrites the program so those regions execute via a delegate call.

In [ ]:
edge_xnn = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()])

# Delegation inspection.
try:
    gm = edge_xnn.exported_program().graph_module
    print("\n=== Lowered graph (XNNPACK) ===")
    print(gm)
    print("\n")

    delegate_op = torch.ops.higher_order.executorch_call_delegate

    delegate_calls = 0
    fallback_ops = 0

    for n in gm.graph.nodes:
        if n.op == "call_function":
            if n.target == delegate_op:
                delegate_calls += 1
            else:
                fallback_ops += 1

    # Heuristic summary of non-delegated call_function ops.
    non_delegate_targets = []
    for n in gm.graph.nodes:
        if n.op == "call_function" and n.target != delegate_op:
            non_delegate_targets.append(str(n.target))

    uniq = sorted(set(non_delegate_targets))
    print("\nSome non-delegated call_function ops (unique, truncated):")
    for t in uniq[:25]:
        print("  ", t)
    if len(uniq) > 25:
        print(f"  ... ({len(uniq)-25} more)")

    print(f"\nXNNPACK delegate call sites : {delegate_calls}")
    print(f"Fallback call_function ops  : {fallback_ops}")

except Exception as e:
    print("Could not print/inspect delegated graph cleanly:", repr(e))
    print("Tip: depending on ExecuTorch version, try:")
    print("  print(edge_xnn.exported_program().graph_module)")

In this example, we are looking at an auto-regressive OPT-125M model that has been exported and partitioned between XNNPACK delegate regions and the default ExecuTorch operator path. Each `LoweredBackendModule` represents a delegated subgraph that will execute using XNNPACK’s optimized kernels, while the remaining operators run through the standard ExecuTorch runtime (i.e., the default ExecuTorch operators backed by the Portable Kernel Library). The delegate call count shows how many distinct regions were offloaded, and the remaining operator count reflects what stayed on the default path. In general, greater delegation is beneficial because XNNPACK provides highly optimized CPU implementations, especially for linear, matmul, convolution, and common elementwise operations. However, there is an important balance in that a few large delegated regions are ideal, whereas many small delegated regions interleaved with default ExecuTorch operators can introduce overhead due to graph fragmentation and repeated transitions between execution domains.

With OPT-125M, we expect to see improvement compared with running entirely through PyTorch or the default ExecuTorch runtime, since the large linear and attention matmuls benefit from XNNPACK acceleration. That said, autoregressive transformer decoders often contain masking logic, reshapes, indexing, and control-style tensor ops that are not always delegated. This can lead to more fragmentation compared to simpler feed-forward architectures. In the upcoming DistilBERT encoder example, delegation will be cleaner because the model is bidirectional and structurally simpler than an autoregressive decoder, although transformer-style models in general can still show mixed partitioning. In contrast, the MobileNet example in Lab 2 will demonstrate very strong and clean delegation. Convolution-heavy CNNs map extremely well to XNNPACK, typically resulting in fewer, larger delegate regions and proportionally fewer default ExecuTorch operators. As a result, CNNs often show the most dramatic performance improvements.

We can now serialize the XNNPACK-delegated program and benchmark it with warmup + `ITERS` runs (average latency), matching the PyTorch benchmarking approach.

In [ ]:
et_xnn = edge_xnn.to_executorch()

pte_xnn = outdir / "opt125m_xnnpack.pte"
with open(pte_xnn, "wb") as f:
    et_xnn.write_to_file(f)

prog_xnn = rt.load_program(pte_xnn)
method_xnn = prog_xnn.load_method("forward")

def et_xnn_forward():
    _ = method_xnn.execute((input_ids, attention_mask))

et_times, et_stats = bench_dist(et_xnn_forward, warmup=WARMUP, iters=ITERS)
print_bench_report("ExecuTorch + XNNPACK", et_stats, WARMUP, ITERS)

plot_time_distribution(pt_times, "PyTorch eager")
plot_time_distribution(et_times, "ExecuTorch + XNNPACK")

pte_port_size = pte_port.stat().st_size / 1e6
pte_xnn_size = pte_xnn.stat().st_size / 1e6

print("\nPTE sizes (MB):")
print("default:", f"{pte_port_size:.2f}")
print("xnnpack :", f"{pte_xnn_size:.2f}")

Enabling XNNPACK delegation increases the .pte size because the delegate backend and associated metadata are included in the program. In practice, this increase is typically modest and still well within the constraints of most embedded and mobile-class environments. The trade-off is generally favorable - a slightly larger program footprint in exchange for significantly improved throughput, lower latency, and better CPU efficiency on supported operators.

If running on the Pi, a moderate speed-up should now be visible. 
> Note - using higher-specification devices such as MacBook Pro M4 may reverse this relationship.

Now we have an FP32 XNNPACK ExecuTorch model. Let's actually try prompting it.

This code loads the XNNPACK-delegated model and performs autoregressive text generation. The prompt is tokenized into input_ids and attention_mask, and at each step the model is invoked with a fixed context window, ctx, left-padded as necessary to match the expected input size.

For each iteration, the model outputs logits for the next token, which are temperature-scaled and sampled using multinomial sampling. The sampled token is appended to the input sequence, and the loop continues for max_new_tokens steps before decoding the final generated text.

In [ ]:
from executorch.extension.pybindings.portable_lib import _load_for_executorch

model = _load_for_executorch("Transformer_Lab_ExecuTorch/executorch_artifacts/opt125m_xnnpack.pte")

# Encode prompt
prompt = "The history of artificial intelligence started with"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

ctx = 128
temperature = 0.8
max_new_tokens = 40

for _ in range(max_new_tokens):
    # Take last ctx tokens.
    in_ids = input_ids[:, -ctx:]
    in_mask = attention_mask[:, -ctx:]

    # Left pad if shorter than ctx.
    if in_ids.shape[1] < ctx:
        pad = ctx - in_ids.shape[1]
        pad_ids = torch.zeros((1, pad), dtype=in_ids.dtype)
        pad_mask = torch.zeros((1, pad), dtype=in_mask.dtype)

        in_ids = torch.cat([pad_ids, in_ids], dim=1)
        in_mask = torch.cat([pad_mask, in_mask], dim=1)

    logits = model([in_ids, in_mask])[0]  

    probs = torch.softmax(logits / temperature, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)

    input_ids = torch.cat([input_ids, next_token], dim=1)
    attention_mask = torch.cat(
        [attention_mask, torch.ones_like(next_token)],
        dim=1
    )

print(tokenizer.decode(input_ids[0], skip_special_tokens=True))

> Note: the output above is generated by OPT-125M, and given the low number of parameters, may appear strange.

## **DistilBERT Encoder Transformers Example**

After demonstrating OPT-125M, we now move to DistilBERT. ExecuTorch is not limited to generative LLMs. OPT-125M is a decoder-only, autoregressive transformer focused on text generation. DistilBERT is an encoder-only transformer designed for inference tasks such as classification, question answering, and semantic similarity.

DistilBERT also reflects more common on-device NLP workloads. Many real-world mobile and edge applications rely on encoder models for tasks such as sentiment analysis, search ranking, and intent detection rather than free-form text generation. These models perform a single forward pass over a fixed-length sequence, making them simpler operationally while still being dominated by the same core building blocks of linear layers, attention matmuls, and feed-forward networks. Again, XNNPACK can accelerate transformer compute, whilst providing a compact, portable artifact and low memory footprint.

Initial setup occurs below. We first load the pre-trained DistilBERT masked language model and its tokenizer. The tokenizer converts raw text into the token IDs and attention masks that the model expects. The model is switched to evaluation mode so it runs in inference-only mode.

We then wrap the model with a small helper module that simplifies its output. Instead of returning predictions for every token in the sequence, we extract only the logits at the `[MASK]` position. This is because, for masked language modeling, the rest of the logits are not used. Finally, we tokenize a sentence containing a single `[MASK]` token and pad it to a fixed length so the model sees consistent input shapes.

In [ ]:
# =========================
# DistilBERT Fill-Mask (MLM) — PyTorch vs ExecuTorch+XNNPACK.
# =========================

from transformers import AutoModelForMaskedLM

# Config.
MODEL_ID = "distilbert-base-uncased"
SEQ_LEN  = 64
DTYPE    = torch.float32
DEVICE   = "cpu"
WARMUP   = 2
ITERS    = 20
TOPK     = 10

# Load tokenizer + MLM model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
hf_mlm = AutoModelForMaskedLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

# Return logits at [MASK] only -> [B, V] (keeps outputs small & stable).
class DistilBertMaskLogits(torch.nn.Module):
    def __init__(self, m, mask_id: int):
        super().__init__()
        self.m = m
        self.mask_id = mask_id

    def forward(self, input_ids, attention_mask):
        logits = self.m(input_ids=input_ids, attention_mask=attention_mask).logits  # [B,T,V].

        # Assume exactly one [MASK] per example:
        mask_pos = (input_ids == self.mask_id).to(torch.int64).argmax(dim=1)        # [B].
        b = torch.arange(input_ids.shape[0], device=input_ids.device)
        return logits[b, mask_pos, :]                                              # [B,V].

wrapped = DistilBertMaskLogits(hf_mlm, tokenizer.mask_token_id).eval()

# Single-mask prompt, fixed-shape to SEQ_LEN.
prompt = f"Books are made for {tokenizer.mask_token}."
enc = tokenizer(
    prompt,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=SEQ_LEN,
)
input_ids = enc["input_ids"].to(DEVICE)
attention_mask = enc["attention_mask"].to(DEVICE)

We now run the benchmarks and display the delegated graph and split of operators between XNNPACK and default.

In [ ]:
# -------------------------
# PyTorch benchmark.
# -------------------------
def pt_forward():
    with torch.inference_mode():
        _ = wrapped(input_ids, attention_mask)

pt_times_db, pt_stats_db = bench_dist(pt_forward, warmup=WARMUP, iters=ITERS)

# -------------------------
# Export once.
# -------------------------
ep = export(wrapped, (input_ids, attention_mask))

outdir = pathlib.Path("Transformer_Lab_ExecuTorch/et_distilbert_artifacts")
outdir.mkdir(exist_ok=True)

# -------------------------
# ExecuTorch + XNNPACK.
# -------------------------
edge_xnn = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()])

# Delegation inspection.
try:
    gm = edge_xnn.exported_program().graph_module
    print("\n=== Lowered graph (XNNPACK) ===")
    print(gm)
    print("\n")

    delegate_op = torch.ops.higher_order.executorch_call_delegate

    delegate_calls = 0
    default_et_ops = 0
    non_delegate_targets = []

    for n in gm.graph.nodes:
        if n.op == "call_function":
            if n.target == delegate_op:
                delegate_calls += 1
            else:
                default_et_ops += 1
                non_delegate_targets.append(str(n.target))

    uniq = sorted(set(non_delegate_targets))
    print("Some default ExecuTorch call_function ops (unique, truncated):")
    for t in uniq[:25]:
        print("  ", t)
    if len(uniq) > 25:
        print(f"  ... ({len(uniq)-25} more)")

    print(f"\nXNNPACK delegate call sites        : {delegate_calls}")
    print(f"Default ExecuTorch call_function ops: {default_et_ops}")

except Exception as e:
    print("Could not print/inspect delegated graph cleanly:", repr(e))
    print("Tip: depending on ExecuTorch version, try:")
    print("  print(edge_xnn.exported_program().graph_module)")

et_xnn = edge_xnn.to_executorch()
pte_xnn = outdir / "distilbert_fillmask_xnnpack.pte"
with open(pte_xnn, "wb") as f:
    et_xnn.write_to_file(f)

prog_xnn = rt.load_program(str(pte_xnn))
method_xnn = prog_xnn.load_method("forward")

def et_xnn_forward():
    _ = method_xnn.execute((input_ids, attention_mask))

# Benchmark.
et_times_db, et_stats_db = bench_dist(et_xnn_forward, warmup=WARMUP, iters=ITERS)

In this run, both models show a broadly similar delegation pattern. You will likely see small variations in delegation due to differences in library versions, build configuration, or environment setup. In our run, DistilBERT has 63 XNNPACK delegate call sites and 283 default ExecuTorch `call_function` ops, while OPT-125M has 134 delegate call sites and 618 default ops. Although OPT-125M has more delegation in absolute terms, it is also a larger graph overall, so the delegate-versus-default split is roughly comparable. A more meaningful distinction is how fragmented the delegated work appears to be. DistilBERT reaches XNNPACK in fewer call sites, 63 vs 134, which can indicate larger delegated partitions per call and potentially lower boundary overhead. In contrast, OPT-125M’s higher number of delegate calls suggests the graph may be divided into more, smaller delegated regions, increasing transitions between XNNPACK and the default ExecuTorch runtime.

However, the number of delegate call sites does not tell the whole story. What actually gets delegated is equally important. If the remaining default ExecuTorch operations are relatively expensive or occur frequently, they may offset the performance gains from XNNPACK, especially at small batch sizes and shorter sequence lengths. Conversely, a model that delegates fewer regions but successfully delegates its computational bottlenecks could see better overall performance. You can examine the printed operator list to understand which operations remain on the default path. In Lab 3, we will introduce the Google Model Explorer, which provides adapters for viewing models across different formats and representations, allowing for deeper inspection of operator composition and delegation patterns.

We now use the ExecuTorch + XNNPACK model to output suggested words to complete the prompt `Books are made for...`, and show the benchmarks:

In [ ]:
# -------------------------
# Sizes + Summary.
# -------------------------
print_bench_report("PyTorch", pt_stats_db, WARMUP, ITERS)
print_bench_report("ExecuTorch + XNNPACK", et_stats_db, WARMUP, ITERS)

plot_time_distribution(pt_times_db, "PyTorch eager")
plot_time_distribution(et_times_db, "ExecuTorch + XNNPACK")

xnn_mb  = pte_xnn.stat().st_size / 1e6

print(f"PTE size (MB) xnnpack: {xnn_mb:.2f}")

In [ ]:
# -------------------------
# Fill-mask demo.
# -------------------------
logits = method_xnn.execute((input_ids, attention_mask))[0]  # [B, V]
topk_ids = torch.topk(logits[0], TOPK).indices.tolist()

print("\nPrompt:", prompt)
print("Top fills:")
for tid in topk_ids:
    tok = tokenizer.decode([tid]).strip()
    print(" -", tok)

# Print full sentences for top-k.
mask_idx = (input_ids[0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
print("\nTop filled sentences:")
for tid in topk_ids[:5]:
    filled = input_ids.clone()
    filled[0, mask_idx] = tid
    print("-", tokenizer.decode(filled[0], skip_special_tokens=True))

> Note: the sentence completions above are from the DistilBERT model, and could appear strange.

## **End of Lab Summary – What Have We Learned?**

In this lab, we explored how modern deep learning models can be exported from PyTorch and deployed efficiently using ExecuTorch and XNNPACK. We demonstrated this across two different transformer architectures: a decoder-only large language model (OPT-125M) and an encoder-only transformer (DistilBERT).

We learned that exporting a model with torch.export produces a stable, static representation that can then be lowered into ExecuTorch’s Edge dialect. From there, the XNNPACK partitioner identifies supported subgraphs and delegates those regions to optimized CPU kernels. The remaining portions of the graph execute using the default ExecuTorch runtime. By inspecting delegate call sites and non-delegated operators, we gained insight into how partitioning works and how graph structure influences performance.

We also observed that performance is not determined solely by how many delegate call sites exist. What matters equally is which operations are delegated and how fragmented the graph becomes. Heavy compute kernels such as linear layers and matrix multiplications benefit most from XNNPACK acceleration, while reshape, masking, indexing, and normalization operations often remain on the default path. The balance between delegated and non-delegated work, along with boundary overhead between execution domains, ultimately determines end-to-end latency.

Importantly, ExecuTorch is not simply “PyTorch, but faster at the edge.” Rather, ExecuTorch enables PyTorch models to be deployed to edge and embedded hardware in production environments that may not have the memory or system resources required to support the full PyTorch runtime and its dependencies. By also leveraging XNNPACK when deploying to CPUs, we can recover many of the optimized compute paths available in PyTorch, and in some cases achieve reduced latency on devices capable of running both frameworks (such as a Raspberry Pi).

In a typical deployment environment, further gains would likely be realized by integrating the ExecuTorch model directly into a C++ application, eliminating notebook and Python runtime overhead. If you want to explore running a model delegated to XNNPACK with CMake, you can follow this PyTorch tutorial [here](https://docs.pytorch.org/executorch/0.3/tutorial-xnnpack-delegate-lowering.html). It also showcases use of the `aot_compiler.py` script, which allows you to perform many of the steps detailed in this lab in a quick and convenient way.

In the next lab, we will extend from transformer-based models to convolutional models (MobileNetV2), allowing us to compare attention-heavy and convolution-heavy workloads under the same deployment framework. We will also introduce quantization, which is commonly used in edge deployments to reduce model size and improve efficiency.

## **References**
- [1] https://engineering.fb.com/2025/07/28/android/executorch-on-device-ml-meta-family-of-apps/#:~:text=Cutouts%20is%20one%20of%20Instagram%E2%80%99s,DAU
